<a href="https://colab.research.google.com/github/JorgeCoronelFuentes/JorgeCoronelFuentes/blob/main/AD2025sentimentanalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers

In [2]:
from transformers import pipeline

# Load the classification pipeline with the specified model
pipe = pipeline("text-classification", model="tabularisai/multilingual-sentiment-analysis")

# Classify a new sentence
sentence = "I love this product! It's amazing and works perfectly."
result = pipe(sentence)

# Print the result
print(result)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/902 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/541M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cpu


[{'label': 'Very Positive', 'score': 0.5922621488571167}]


In [3]:
import pandas as pd

In [4]:
dt=pd.read_excel("/content/Prueba_sentiment_astrazeneca_report.xlsx")

In [14]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd # Import pandas to use isna()

model_name = "tabularisai/multilingual-sentiment-analysis"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

def predict_sentiment(texts):
    sentiments = []
    sentiment_map = {0: "Very Negative", 1: "Negative", 2: "Neutral", 3: "Positive", 4: "Very Positive"}
    for text in texts:
        if pd.isna(text):
            sentiments.append("Unknown") # Or handle as appropriate, e.g., skip
            continue
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
        with torch.no_grad():
            outputs = model(**inputs)
        probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
        sentiments.append(sentiment_map[torch.argmax(probabilities, dim=-1).item()])
    return sentiments

In [11]:
texts=dt['Contenido']

In [12]:
print(texts)

0     Empresas líderes como Genentech y AstraZeneca ...
1     Sin embargo, empresas como Genentech y AstraZe...
2     Sin embargo, empresas de renombre como Genente...
3     No obstante, empresas líderes del sector como ...
4     Sin embargo, empresas como Genentech y AstraZe...
5                                                   NaN
6     Sin embargo, empresas como Genentech y AstraZe...
7     [...] Ha colaborado para INCAE Business School...
8     [...] de la FIFA La AMMC publica el 12.º númer...
9     [...] Ha colaborado para INCAE Business School...
10    En AstraZeneca , todo lo que hacemos está resp...
11    AstraZeneca es un lugar donde la libertad de p...
12    [...] 30 de julioEuropa: Expectativas de infla...
13    Consulta toda la información sobre AstraZeneca...
14    Qué es AstraZeneca ? AstraZeneca es una compañ...
15    Nuestra prioridad en AstraZeneca es la salud y...
16    [...] Ha colaborado para INCAE Business School...
17    Los científicos de AstraZeneca siguen tran

In [16]:
dt['Sentiment_Predicted'] = predict_sentiment(texts)
display(dt.head())

,Unnamed: 0,ID,Fecha,Hrs,Título,Contenido,Fuente,Dominio,Categoría,Sentimiento,Etiquetas,Sentiment_Predicted
0,NaN,88796495135,2025-07-28,13:07,Avances en el Descubrimiento de Fármacos: Impl...,Empresas líderes como Genentech y AstraZeneca ...,https://andaluciainforma.com/avances-en-el-des...,andaluciainforma.com,Noticias,0.0,NaN,Neutral
1,NaN,88796495149,2025-07-28,12:40,Desarrollo de un Asistente de Investigación en...,"Sin embargo, empresas como Genentech y AstraZe...",https://www.opensecurity.es/desarrollo-de-un-a...,opensecurity.es,Noticias,0.0,NaN,Neutral
2,NaN,88796495164,2025-07-28,12:38,Desarrollo de un Asistente de Investigación en...,"Sin embargo, empresas de renombre como Genente...",https://noticias.ai/desarrollo-de-un-asistente...,noticias.ai,Noticias,0.0,NaN,Neutral
3,NaN,88796495122,2025-07-28,12:28,Optimización del Descubrimiento de Fármacos: I...,"No obstante, empresas líderes del sector como ...",https://noticias.madrid/optimizacion-del-descu...,noticias.madrid,Noticias,0.0,NaN,Very Positive
4,NaN,88796495155,2025-07-28,12:22,Desarrollo de un Asistente de Investigación en...,"Sin embargo, empresas como Genentech y AstraZe...",https://www.redes-sociales.com/desarrollo-de-u...,redes-sociales.com,Noticias,0.0,NaN,Neutral


In [9]:
print(texts[0])

Empresas líderes como Genentech y AstraZeneca están recurriendo a la inteligencia artificial (IA) para acelerar procesos que tradicionalmente consumen tiempo y recursos. [...] Empresas líderes como Genentech y AstraZeneca están recurriendo a la intel


In [10]:
pipe(texts[0])

[{'label': 'Neutral', 'score': 0.3517685532569885}]

In [17]:
sentiment_frequencies = dt['Sentiment_Predicted'].value_counts()
display(sentiment_frequencies)

,count
Sentiment_Predicted,
Neutral,30
Very Positive,12
Positive,6
Unknown,3
